# 00 · Quickstart

Build a synthetic project, open it, and look at what came in.

Everything in these notebooks runs on generated data, so they work on a fresh
checkout with no files to download and nothing to configure.

**What a project is**

| | |
|---|---|
| `MetaData/*.py` | sample-keyed dicts — the authored source of truth |
| `<Modality>Data/<SampleID>/` | data nobody wrote into a dict |
| `.nanoorganizer/` | the canonical store the package writes |

The demo has a hidden control variable — a synthesis temperature — that moves
both the spectral band and the particle size. The point of the pipeline is to
recover it. See `docs/sample_model.md` for the model itself.

In [1]:
from pathlib import Path

from NanoOrganizer import open_project
from NanoOrganizer.demo import build_demo_project

PROJECT = Path.home() / "DemoProject"

# Generated on demand — nothing is committed to the repository.
if not (PROJECT / "MetaData").exists():
    build_demo_project(PROJECT)

wb = open_project(PROJECT)
print(wb.summary())

DemoProject: 6 samples, 12 measurements (12 readable here, 0 not mounted).
Modalities: uvvis, tem
Root: /home/yuzhang/DemoProject


## What came in

One row per sample, with the authored parameters flattened into dotted
columns.

In [2]:
table = wb.table()
print(f"{len(table)} samples x {len(table.columns)} columns")
print("modalities:", wb.project.modalities())
print("groups:   ", wb.project.groups())

table[[c for c in table.columns if "conditions" in c or c.endswith("status")]]

6 samples x 36 columns
modalities: ['uvvis', 'tem']
groups:    ['curve', 'image']


,synthesis.status,synthesis.synthesis_batch.status,synthesis.conditions.temperature_C,synthesis.conditions.stir_rate_rpm,synthesis.conditions.solvent,synthesis.conditions.hold_time_min
sample_id,,,,,,
Sample000001,done,done,60.0,450,water,30.0
Sample000002,done,done,70.0,500,water,30.0
Sample000003,done,done,80.0,400,water,30.0
Sample000004,done,done,90.0,450,water,30.0
Sample000005,done,done,100.0,500,water,30.0
Sample000006,error,error,110.0,400,water,30.0


## Measurements

A *measurement* references files without reading them. Two arrived per sample
by different routes: the spectra from a glob in the metadata, the micrographs
from the `TEMData/<SampleID>/` folder convention.

In [3]:
wb.table(level="measurement")[
    ["sample_id", "modality_label", "group", "stage", "n_paths"]]

,sample_id,modality_label,group,stage,n_paths
0,Sample000001,UV-Vis,curve,synthesis,0
1,Sample000001,TEM,image,characterization,3
2,Sample000002,UV-Vis,curve,synthesis,0
3,Sample000002,TEM,image,characterization,3
4,Sample000003,UV-Vis,curve,synthesis,0
5,Sample000003,TEM,image,characterization,3
6,Sample000004,UV-Vis,curve,synthesis,0
7,Sample000004,TEM,image,characterization,3
8,Sample000005,UV-Vis,curve,synthesis,0
9,Sample000005,TEM,image,characterization,3


## Is the data reachable?

Paths are stored exactly as recorded and resolved per machine, so a project
whose data is not mounted still browses — it reports as unreadable rather than
breaking.

In [4]:
report = wb.available()
print(f"{report['n_available']}/{report['n_measurements']} measurements readable here")

for sample in report["samples"]:
    for m in sample["measurements"]:
        if not m["available"]:
            print("  missing:", m["measurement_id"], m["missing"][:1])

12/12 measurements readable here


If data lives somewhere else on this machine, map the prefix once
rather than rewriting the metadata — that keeps the store portable:

```python
wb.project.add_alias("/instrument/share", ["/mnt/instrument"])
```

## Save

In [5]:
wb.save()

PosixPath('/home/yuzhang/DemoProject/.nanoorganizer/samples.json')